In [3]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ibm import WatsonxEmbeddings, ChatWatsonx
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_ollama import OllamaEmbeddings
from langchain_core.runnables import RunnablePassthrough
import re
from pprint import pprint

import bs4

In [29]:
import os

from dotenv import load_dotenv


load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watson_ai_url = os.getenv("WATSONX_URL")

watson_llm = ChatWatsonx(
    apikey=apikey,
    project_id=project_id,
    watsonx_url=watson_ai_url,
    model_id="ibm/granite-4-h-small",
    max_tokens=2000,
    params={"temperature": 0},
)

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url=f"{watson_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    params={"temperature": 0},
)

In [26]:
web_loader = WebBaseLoader(
    web_paths=["https://n.news.naver.com/article/015/0005293942?type=main"],
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(attrs={"id": ["title_area", "dic_area"]})
    ),
)


web_docs = web_loader.load()
pprint(web_docs)

content = web_docs[0].page_content

[Document(metadata={'source': 'https://n.news.naver.com/article/015/0005293942?type=main'}, page_content='강남 아파트 팔아 \'ETF 투자\'…금감원장, 얼마나 벌었을까\n\n\n\n\n이찬진 금융감독원장 / 사진=연합뉴스코스피가 상한가를 이어가면서 강남 아파트를 매각한 후 ETF에 투자한 이찬진 금융감독원장의 수익률에 이목이 쏠리고 있다. 일각에서는 이 원장이 매수 시점 대비 약 2.5배 이상의 \'잭팟\'에 가까운 수익률을 기록했을 것이란 추정도 나온다.이 원장은 지난해 10월 말 보유 중이던 서울 서초구 우면동 소재 아파트 한 채를 급매한 후 국내 지수 추종 ETF를 매수했다. 이 원장은 해당 아파트를 배우자와 공동명의로 보유했는데, 다주택자 논란이 일자 처분했다. 이 원장의 아파트는 기존 호가였던 22억원보다 4억원 낮춘 18억원이었다.해당 아파트 매도와 관련해 이 원장은 국정감사에서 "공간이 좁아져 고통이 조금 있는 부분이지만, 공직자라는 신분을 감안해 한 채를 처분하고 정리하겠다"고 말했다.이 원장은 매매 체결 당일 KB증권 영업점을 직접 찾아 계약금으로 코스피, 코스닥 지수 추종 상품에 가입했다. 구체적인 금액과 종목은 알려지지 않았지만 계약금 전액을 투자했다면, 당시 매수 금액을 고려했을 때 2억원 상당으로 추정된다.코스피200 지수를 추종하는 대표적인 ETF인 KODEX 200의 주가 상승률은 이 원장 매수 시점부터 전날까지 151.55%였다. 비슷한 코스피 지수 추종 ETF인 TIGER 200, ACE 200, RISE 200 등도 같은 기간 수익률이 150%를 웃돌고 있다. 계약금 2억원을 전액 투자했다고 가정하면 평가 차익만 3억원에 이르는 셈이다.코스닥150 지수를 추종하는 ETF들의 경우 이 원장 매매 시점부터 약 20% 상승률을 기록 중이다. 코스피와 코스닥 대표 지수에 반반 투자했다고 가정한다면 평가 차익은 약 1억7000만원에 육박한다.이 원장은 

In [27]:
lines = [line for line in content.split("\n") if line.strip()]
title = lines[0]


web_docs[0].page_content = "\n\n".join(lines[0:2])

web_docs[0].metadata["title"] = title

pprint(web_docs[0].page_content)

pprint(web_docs[0].metadata)

("강남 아파트 팔아 'ETF 투자'…금감원장, 얼마나 벌었을까\n"
 '\n'
 '이찬진 금융감독원장 / 사진=연합뉴스코스피가 상한가를 이어가면서 강남 아파트를 매각한 후 ETF에 투자한 이찬진 금융감독원장의 수익률에 '
 "이목이 쏠리고 있다. 일각에서는 이 원장이 매수 시점 대비 약 2.5배 이상의 '잭팟'에 가까운 수익률을 기록했을 것이란 추정도 "
 '나온다.이 원장은 지난해 10월 말 보유 중이던 서울 서초구 우면동 소재 아파트 한 채를 급매한 후 국내 지수 추종 ETF를 매수했다. '
 '이 원장은 해당 아파트를 배우자와 공동명의로 보유했는데, 다주택자 논란이 일자 처분했다. 이 원장의 아파트는 기존 호가였던 22억원보다 '
 '4억원 낮춘 18억원이었다.해당 아파트 매도와 관련해 이 원장은 국정감사에서 "공간이 좁아져 고통이 조금 있는 부분이지만, 공직자라는 '
 '신분을 감안해 한 채를 처분하고 정리하겠다"고 말했다.이 원장은 매매 체결 당일 KB증권 영업점을 직접 찾아 계약금으로 코스피, 코스닥 '
 '지수 추종 상품에 가입했다. 구체적인 금액과 종목은 알려지지 않았지만 계약금 전액을 투자했다면, 당시 매수 금액을 고려했을 때 2억원 '
 '상당으로 추정된다.코스피200 지수를 추종하는 대표적인 ETF인 KODEX 200의 주가 상승률은 이 원장 매수 시점부터 전날까지 '
 '151.55%였다. 비슷한 코스피 지수 추종 ETF인 TIGER 200, ACE 200, RISE 200 등도 같은 기간 수익률이 '
 '150%를 웃돌고 있다. 계약금 2억원을 전액 투자했다고 가정하면 평가 차익만 3억원에 이르는 셈이다.코스닥150 지수를 추종하는 '
 'ETF들의 경우 이 원장 매매 시점부터 약 20% 상승률을 기록 중이다. 코스피와 코스닥 대표 지수에 반반 투자했다고 가정한다면 평가 '
 "차익은 약 1억7000만원에 육박한다.이 원장은 지난 2월 열린 기자간담회에서 '아파트 매각 대금으로 ETF를 추가 매수할 계획이 "
 '있는지\' 묻는 질문에

In [32]:
from langchain_classic.prompts import ChatPromptTemplate


# 임베딩 모델 최대 입력이 512 토큰이므로, 그 아래로 청크 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
)
splits = text_splitter.split_documents(web_docs)

print(f"분할된 청크 수: {len(splits)}")

vectorstore = FAISS.from_documents(
    documents=splits, embedding=watson_embedding
)

# 디버깅용
vectorstore.similarity_search("금감원장은 어느정도의 이윤을 챙겼는가", k=3)

retriever = vectorstore.as_retriever(k=3)

rag_prompt = ChatPromptTemplate.from_template(
    template="""\
        당신은 뉴스 기사 QA 시스템 입니다.
        
        Rules:
        1. 제공된 기사 내용만 사용하세요.
        2. 기사 내용에 없는 정보는 추측하지 말고, 기사에서 확인할 수 없다고 답하세요.
        
        뉴스 기사:
        {context}

        질문:
        {question}

        답변:
    """
)

분할된 청크 수: 5


In [37]:
from langchain_core.output_parsers import StrOutputParser


def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])


chain = {
    "context": retriever | format_docs,
    "question": RunnablePassthrough()
} | rag_prompt | watson_llm | StrOutputParser()

question = "이 원장의 아파트 매각과 관련해 국정감사에서 어떤 설명을 했나요?"

answer = chain.invoke(question)

print(answer)

이찬진 금융감독원장은 국정감사에서 아파트 매각에 대해 "공간이 좁아져 고통이 조금 있는 부분이지만, 공직자라는 신분을 감안해 한 채를 처분하고"라고 설명했습니다.
